# JAX Internals and The Execution Model

## Setup

In [1]:
# !pip install jax

In [2]:
import jax
import jax.numpy as jnp

In [3]:
print(f"JAX version: {jax.__version__}")
print(f"Available devices: {jax.devices()}")

JAX version: 0.10.0
Available devices: [CpuDevice(id=0)]


In [4]:
x = jnp.array(1.0)
print(x.dtype)

float32


## Our first `Jaxpr`


In [5]:
def f(x):
    return jnp.sin(x) + x * 3.0

In [6]:
jaxpr = jax.make_jaxpr(f)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = mul a 3.0:f32[]
    d:f32[] = add b c
  in (d,) }


## Anatomy of a `Jaxpr`

In [7]:
print(f"jaxpr: {jaxpr}")
print(f"type of jaxpr: {type(jaxpr).__name__}")

print(f"jaxpr.jaxpr: {jaxpr.jaxpr}")
print(f"type of jaxpr.jaxpr: {type(jaxpr.jaxpr).__name__}")

jaxpr: { lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = mul a 3.0:f32[]
    d:f32[] = add b c
  in (d,) }
type of jaxpr: ClosedJaxpr
jaxpr.jaxpr: { lambda ; a:f32[]. let
    b:f32[] = sin a
    c:f32[] = mul a 3.0:f32[]
    d:f32[] = add b c
  in (d,) }
type of jaxpr.jaxpr: Jaxpr


### General structure of a `Jaxpr`

```
jaxpr ::= { lambda <constvars> ; <invars>. 
            let <equations>
            in  <expressions> }
```

### Concrete example

In [8]:
def f(x, y):
    a = jnp.sin(x)
    b = y * 3.0
    return a + b, a, b

In [9]:
jaxpr = jax.make_jaxpr(f)(jnp.array(1.0), jnp.array(2.0))
print(jaxpr)

{ lambda ; a:f32[] b:f32[]. let
    c:f32[] = sin a
    d:f32[] = mul b 3.0:f32[]
    e:f32[] = add c d
  in (e, c, d) }


In [10]:
print(f"Input variables (invars): {jaxpr.jaxpr.invars}")
print(f"Output variables (outvars): {jaxpr.jaxpr.outvars}")
print(f"Equations: {jaxpr.jaxpr.eqns}")

Input variables (invars): [Var(id=2287455770944):float32[], Var(id=2287455772032):float32[]]
Output variables (outvars): [Var(id=2287455774144):float32[], Var(id=2287455770816):float32[], Var(id=2287455770624):float32[]]
Equations: [a:f32[] = sin b, a:f32[] = mul b 3.0:f32[], a:f32[] = add b c]


In [11]:
for eqn in jaxpr.jaxpr.eqns:
    print(
        f" primitive: {eqn.primitive.name}",
        f"| inputs: {eqn.invars}",
        f"| outvars: {eqn.outvars}",
    )

 primitive: sin | inputs: [Var(id=2287455770944):float32[]] | outvars: [Var(id=2287455770816):float32[]]
 primitive: mul | inputs: [Var(id=2287455772032):float32[], Literal(3.0)] | outvars: [Var(id=2287455770624):float32[]]
 primitive: add | inputs: [Var(id=2287455770816):float32[], Var(id=2287455770624):float32[]] | outvars: [Var(id=2287455774144):float32[]]


## Constants

In [12]:
matrix = jnp.ones((3, 3))


def f(x):
    return matrix @ x

In [13]:
jaxpr = jax.make_jaxpr(f)(jnp.ones(3))
print(jaxpr)
print()
print(f"consts: {jaxpr.consts}")

{ lambda a:f32[3,3]; b:f32[3]. let
    c:f32[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] a b
  in (c,) }

consts: [Array([[1., 1., 1.],
       [1., 1., 1.],
       [1., 1., 1.]], dtype=float32)]


In [14]:
for var, val in zip(jaxpr.jaxpr.constvars, jaxpr.consts, strict=True):
    print(f"{var} -> {val.shape}")

Var(id=2287455916352):float32[3,3] -> (3, 3)


In [15]:
scale = jnp.array(3.0)  # JAX array


def g(x):
    return x * scale


print("Closing over a JAX array:")
print(jax.make_jaxpr(g)(jnp.array(1.0)))

Closing over a JAX array:
{ lambda a:f32[]; b:f32[]. let c:f32[] = mul b a in (c,) }


In [16]:
py_scale = 3.0  # Python float


def h(x):
    return x * py_scale


print("Closing over a Python float:")
print(jax.make_jaxpr(h)(jnp.array(1.0)))

Closing over a Python float:
{ lambda ; a:f32[]. let b:f32[] = mul a 3.0:f32[] in (b,) }


## A First Look at Primitives

```
equation ::= let <output vars> = <primitive> [<params>] <input exprs>
```

## Evaluating a Jaxpr

In [17]:
result = jax.core.eval_jaxpr(jaxpr.jaxpr, jaxpr.consts, jnp.ones(3))
print(f"eval_jaxpr result: {result}")
print(f"Direct call result: {f(jnp.ones(3))}")

eval_jaxpr result: [Array([3., 3., 3.], dtype=float32)]
Direct call result: [3. 3. 3.]


## Tracing with Abstract Values

In [18]:
def spy_on_tracing(x):
    print(f"Type of x during tracing: {type(x).__name__}")
    print(f"Value of x during tracing: {x}")
    return 2.0 * x

In [19]:
jax.make_jaxpr(spy_on_tracing)(jnp.array([6.0, 12.0]))

Type of x during tracing: DynamicJaxprTracer
Value of x during tracing: JitTracer(float32[2])


{ lambda ; a:f32[2]. let b:f32[2] = mul 2.0:f32[] a in (b,) }

## Python Control Flow on Dynamic Values

In [20]:
def broken_function(x):
    if x > 0:
        return x * 2.0
    else:
        return x

In [21]:
try:
    jax.make_jaxpr(broken_function)(jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerBoolConversionError
Attempted boolean conversion of traced array with shape bool[].
The error occurred while tracing the function broken_function at C:\Users\user\AppData\Local\Temp\ipykernel_33520\1210078963.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument x.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerBoolConversionError


In [22]:
def fixed_function(x):
    return jax.lax.cond(x > -1, lambda x: x * 2.0, lambda x: x, x)

In [23]:
jaxpr = jax.make_jaxpr(fixed_function)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f32[]. let
    b:bool[] = gt a -1.0:f32[]
    c:i32[] = convert_element_type[new_dtype=int32 weak_type=False] b
    d:f32[] = cond[
      branches=(
        { lambda ; e:f32[]. let  in (e,) }
        { lambda ; f:f32[]. let g:f32[] = mul f 2.0:f32[] in (g,) }
      )
    ] c a
  in (d,) }


## Python Loops Are Unrolled at Trace Time

In [24]:
def loop_function(x):
    for _ in range(3):
        x = jnp.tanh(x)
    return x


jaxpr = jax.make_jaxpr(loop_function)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f32[]. let
    b:f32[] = tanh a
    c:f32[] = tanh b
    d:f32[] = tanh c
  in (d,) }


In [25]:
def loop_function_alternative(x):
    x = jnp.tanh(x)
    x = jnp.tanh(x)
    x = jnp.tanh(x)
    return x


jaxpr = jax.make_jaxpr(loop_function_alternative)(jnp.array(1.0))
print(jaxpr)

{ lambda ; a:f32[]. let
    b:f32[] = tanh a
    c:f32[] = tanh b
    d:f32[] = tanh c
  in (d,) }


In [26]:
def loop_function_broken(y, x):
    for _ in range(y):
        x = jnp.tanh(x)
    return x


try:
    jax.make_jaxpr(loop_function_broken)(3, jnp.array(1.0))
except Exception as e:
    print(type(e).__name__)
    print(e)

TracerIntegerConversionError
The __index__() method was called on traced array with shape int32[]
The error occurred while tracing the function loop_function_broken at C:\Users\user\AppData\Local\Temp\ipykernel_33520\2190278666.py:1 for jit. This concrete value was not available in Python because it depends on the value of the argument y.
See https://docs.jax.dev/en/latest/errors.html#jax.errors.TracerIntegerConversionError


In [27]:
def dynamic_loop(x, n):
    def body(_, val):
        return jnp.tanh(val)

    return jax.lax.fori_loop(0, n, body, x)

In [28]:
jaxpr = jax.make_jaxpr(dynamic_loop)(jnp.array(1.0), 10000)
print(jaxpr)

{ lambda ; a:f32[] b:i32[]. let
    _:i32[] _:i32[] c:f32[] = while[
      body_jaxpr={ lambda ; d:i32[] e:i32[] f:f32[]. let
          g:i32[] = add d 1:i32[]
          h:f32[] = tanh f
        in (g, e, h) }
      body_nconsts=0
      cond_jaxpr={ lambda ; i:i32[] j:i32[] k:f32[]. let
          l:bool[] = lt i j
        in (l,) }
      cond_nconsts=0
    ] 0:i32[] b a
  in (c,) }


## Static vs. Dynamic Values

In [29]:
def uses_n(x, n):
    return x * n

In [30]:
jaxpr_dynamic = jax.make_jaxpr(uses_n)(jnp.array(1.0), jnp.array(2.0))
print("n dynamic")
print(jaxpr_dynamic)

n dynamic
{ lambda ; a:f32[] b:f32[]. let c:f32[] = mul a b in (c,) }


In [31]:
jaxpr_static = jax.make_jaxpr(uses_n, static_argnums=1)(jnp.array(1.0), 2.0)
print("n static")
print(jaxpr_static)

n static
{ lambda ; a:f32[]. let b:f32[] = mul a 2.0:f32[] in (b,) }


## Side Effects at Trace Time vs. Runtime

In [32]:
counter = [0]


def side_effect_function(x):
    counter[0] += 1
    print(f"counter[0]: {counter[0]}")
    return x * 2.0

In [33]:
print("=== Tracing ===")
jaxpr = jax.make_jaxpr(side_effect_function)(jnp.array(1.0))
print(jaxpr)
print()
print(f"Counter value: {counter[0]}")

=== Tracing ===
counter[0]: 1
{ lambda ; a:f32[]. let b:f32[] = mul a 2.0:f32[] in (b,) }

Counter value: 1


In [34]:
print("=== Normal call ===")
_ = side_effect_function(jnp.array(1.0))
print(f"Counter value: {counter[0]}")

=== Normal call ===
counter[0]: 2
Counter value: 2


## Reading a Real Jaxpr

In [35]:
def softmax(x):
    x_shifted = x - jnp.max(x)
    e_x = jnp.exp(x_shifted)
    return e_x / jnp.sum(e_x)

In [36]:
jaxpr = jax.make_jaxpr(softmax)(jnp.ones(4))
print(jaxpr)

{ lambda ; a:f32[4]. let
    b:f32[] = reduce_max[axes=(0,)] a
    c:f32[4] = sub a b
    d:f32[4] = exp c
    e:f32[] = reduce_sum[axes=(0,) out_sharding=None] d
    f:f32[4] = div d e
  in (f,) }


## Pytrees: Structured Inputs and Outputs

In [37]:
params = {"w": jnp.ones((3, 2)), "b": jnp.zeros((3,))}


def linear(params, x):
    return params["w"] @ x + params["b"]

In [38]:
jaxpr = jax.make_jaxpr(linear)(params, jnp.ones((2)))
print(jaxpr)

{ lambda ; a:f32[3] b:f32[3,2] c:f32[2]. let
    d:f32[3] = dot_general[
      dimension_numbers=(([1], [0]), ([], []))
      preferred_element_type=float32
    ] b c
    e:f32[3] = add d a
  in (e,) }


In [39]:
leaves, treedef = jax.tree_util.tree_flatten(params)
print(f"Leaves (shapes): {[l.shape for l in leaves]}")
print(f"Treedef: {treedef}")

Leaves (shapes): [(3,), (3, 2)]
Treedef: PyTreeDef({'b': *, 'w': *})


In [40]:
params_back = treedef.unflatten(leaves)
print(f"params_back: {params_back}")

params_back: {'b': Array([0., 0., 0.], dtype=float32), 'w': Array([[1., 1.],
       [1., 1.],
       [1., 1.]], dtype=float32)}


In [41]:
scaled_params = jax.tree_util.tree_map(lambda p: p * 0.01, params)
print(f"scaled_params: {scaled_params}")

scaled_params: {'b': Array([0., 0., 0.], dtype=float32), 'w': Array([[0.01, 0.01],
       [0.01, 0.01],
       [0.01, 0.01]], dtype=float32)}


## All Transformations On the Same Jaxpr (preview)

In [42]:
def f(x):
    return jnp.sum(x**2)

In [43]:
print("=== Jaxpr of f ===")
print(jax.make_jaxpr(f)(jnp.ones((3))))

=== Jaxpr of f ===
{ lambda ; a:f32[3]. let
    b:f32[3] = integer_pow[y=2] a
    c:f32[] = reduce_sum[axes=(0,) out_sharding=None] b
  in (c,) }


In [44]:
print("=== Jaxpr of grad(f) ===")
print(jax.make_jaxpr(jax.grad(f))(jnp.ones((3))))

=== Jaxpr of grad(f) ===
{ lambda ; a:f32[3]. let
    b:f32[3] = integer_pow[y=2] a
    c:f32[3] = integer_pow[y=1] a
    d:f32[3] = mul 2.0:f32[] c
    _:f32[] = reduce_sum[axes=(0,) out_sharding=None] b
    e:f32[3] = broadcast_in_dim 1.0:f32[]
    f:f32[3] = mul e d
  in (f,) }


In [45]:
print("=== Jaxpr of vmap(f) ===")
print(jax.make_jaxpr(jax.vmap(f))(jnp.ones((4, 3))))

=== Jaxpr of vmap(f) ===
{ lambda ; a:f32[4,3]. let
    b:f32[4,3] = integer_pow[y=2] a
    c:f32[4] = reduce_sum[axes=(np.int64(1),) out_sharding=None] b
  in (c,) }
